# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bajwaycodes/Kashif-Working-Repo/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring**

I'm picking this lane because I've already spent time in the starter dataset building
exactly this: a hand-written rule (stale × visible pages), a readable decision tree, and
a client-holdout validation pass. That work surfaced a real, concrete problem worth
seven more weeks on — a plain rule and a small tree disagree with each other depending on
which clients get held out, which means the "right page to review first" question isn't
settled yet on this data. I want to keep pushing on *which* signals actually generalize
across clients, not just which ones look good in-sample.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question: decision, action, cost of a wrong call

**Question:** Which content pages should a content editor review first for refresh,
given limited review capacity?

**Decision this improves:** Right now an editor has to pick which pages to look at
first with no ranked, evidence-backed queue — they'd otherwise work by gut feel or
alphabetically. This project produces a ranked list with reasons attached.

**Who acts, and how:** A content editor or SEO strategist works down the ranked queue,
opening the top N pages (as many as their week allows) and deciding whether to refresh,
expand, or leave each one — using the reason codes as a starting hypothesis, not a verdict.

**Cost of a wrong call:**
- **False positive** (flagged as worth reviewing, but it wasn't really declining):
  wastes an editor's time — cheap per-instance, but adds up across a queue.
- **False negative** (a genuinely declining page never surfaces): a real traffic loss
  goes unnoticed and unaddressed, which is the more expensive mistake since it compounds
  the longer it's missed.

Because false negatives cost more than false positives, precision at the top of the
queue matters, but recall of the truly declining pages matters too — I'll track both,
not just one.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Number 1: scale and declining rate
n_pages = df.shape[0]
n_clients = df["client_id"].nunique()
declining_rate = df["is_declining_label"].mean()
print(f"Pages: {n_pages:,}  |  Clients: {n_clients}  |  Declining rate: {declining_rate:.1%}")

# Number 2: client concentration — why validation design matters here
client_counts = df["client_id"].value_counts()
top_client_share = client_counts.iloc[0] / n_pages
print(f"Largest client alone holds {top_client_share:.1%} of all pages "
      f"({client_counts.iloc[0]:,} of {n_pages:,})")

# Number 3: baseline vs learned model, from the committed pipeline output
# (verified in outputs/model_report.md / outputs/model_results.json)
print("\nFrom the starter pipeline's own verified results (outputs/model_report.md):")
print("  baseline rules       Precision@50 = 0.240  (~12 of top 50 right)")
print("  random forest        Precision@50 = 0.740  (~37 of top 50 right)")

Pages: 30,000  |  Clients: 32  |  Declining rate: 54.2%
Largest client alone holds 23.4% of all pages (7,008 of 30,000)

From the starter pipeline's own verified results (outputs/model_report.md):
  baseline rules       Precision@50 = 0.240  (~12 of top 50 right)
  random forest        Precision@50 = 0.740  (~37 of top 50 right)


**What these numbers say:**

- **30,000 pages across 32 clients**, with roughly a 54% declining rate in this slice —
  plenty of positive examples to learn from, not a rare-event problem.
- **Client concentration is severe** — the single largest client holds well over a fifth
  of all pages. This isn't a footnote; it means any train/test split *must* be grouped by
  client (never a plain random row split), or the reported precision is fiction.
- **The committed pipeline result** shows a random forest roughly tripling the baseline
  rule's Precision@50 (0.240 → 0.740) on this dataset. That's a strong signal this lane is
  worth building on — but it's an in-sample-adjacent number from one run, and my own
  client-holdout experiments already show it swings a lot by seed. That instability *is*
  part of what weeks 2–7 need to resolve, not paper over.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work will be able to say:**
- *Observed*: which signals (staleness, visibility, position, CTR, engagement) are
  associated with pages later labeled as declining, in this dataset.
- *Directional*: pages with certain combinations of signals appear more often among
  declining pages than others — a tendency, not a certainty for any one page.
- *Decision-support*: a ranked queue that helps a human reviewer spend limited time on
  the most promising candidates first, with reason codes they can inspect and override.

**What this work will never claim:**
- **Not causal proof.** I cannot say refreshing a page *caused* it to recover — that
  needs an actual experiment (e.g., an A/B test on real refresh actions), which this
  data doesn't provide.
- **Not "predicting Google."** I'm not modeling a search engine's ranking algorithm —
  only observable signals (impressions, clicks, position, engagement) that FlyRank's
  own systems already measure.
- **Not a guarantee for any individual page.** A high score means "worth a look," not
  "will definitely improve if edited."
- **Not validated on the full warehouse yet.** Everything above comes from a 30,000-row
  anonymized starter slice — any number here needs to be re-earned on the larger
  warehouse release before I'd trust it for a real recommendation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.